In [ ]:
from pathlib import Path
import warnings
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.impute import SimpleImputer
from scipy.interpolate import UnivariateSpline
import json, warnings
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import shap
warnings.filterwarnings('ignore')


def find_project_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'database.xlsx').exists():
            return candidate
    raise FileNotFoundError('Could not locate database.xlsx in the current directory or its parents.')

ROOT = find_project_root()
OUT = ROOT / 'outputs' / 'figures'
OUT.mkdir(parents=True, exist_ok=True)
DATA_PATH = ROOT / 'database.xlsx'
MODEL_PATH = ROOT / 'catboost_nce.cbm'
SHAP_PATH = ROOT / 'shap_values.npy'
INTERACTION_PATH= ROOT / 'shap_interaction_values.npy'
FEATURE_COLUMNS = ['AM', 'AMc', 'Pr', 'Prc', 'Sp', 'Sp2', 'MSP', 'PM', 'CT', 'Ct', 'RT', 'Rt', 'RH', 'T', 'P', 'GHSV', 'Inert', 'H2/CO2']
CATEGORICAL = ['AM', 'Pr', 'Sp', 'Sp2', 'PM']
CONTINUOUS = [feature for feature in FEATURE_COLUMNS if feature not in CATEGORICAL]
TARGET = 'Rate'


In [ ]:
from matplotlib.lines import Line2D

FIG_WIDTH_MM,FIG_HEIGHT_MM,POINT_SIZE,POINT_ALPHA,JITTER_SD,BAR_HEIGHT,BAR_ALPHA,DPI=183,170,8.0,.38,.105,.72,.24,600
SHAP_FEATURES=FEATURE_COLUMNS
SHAP_CATEGORICAL=['AM','Pr','Sp','Sp2','PM']
SHAP_CONTINUOUS=[f for f in SHAP_FEATURES if f not in SHAP_CATEGORICAL]
SHAP_GROUPS={'Catalyst composition':['AM','AMc','Pr','Prc','Sp','Sp2','MSP'],'Preparation':['PM','CT','Ct','RT','Rt','RH'],'Operating condition':['T','P','GHSV','Inert','H2/CO2']}
SHAP_GROUP_COLORS={'Operating condition':'#DB3124','Catalyst composition':'#FFDF92','Preparation':'#4B74B2'}
SHAP_GROUP_TEXT_COLORS={'Operating condition':'#C82423','Catalyst composition':'#A66A00','Preparation':'#365F91'}
SHAP_DISPLAY={'H2/CO2':r'H$_2$/CO$_2$'}

mpl.rcParams.update({'font.family':'serif','font.serif':['Times New Roman','DejaVu Serif'],'mathtext.fontset':'stix','font.size':9.5,'axes.labelsize':11,'axes.titlesize':11,'xtick.labelsize':9.5,'ytick.labelsize':10,'axes.linewidth':.75,'axes.spines.top':False,'axes.spines.right':False,'legend.frameon':False,'legend.fontsize':9,'svg.fonttype':'none','pdf.fonttype':42,'savefig.facecolor':'white'})

if shap_values.shape!=(len(X),len(SHAP_FEATURES)): raise ValueError(f'SHAP shape {shap_values.shape} does not match data shape {(len(X),len(SHAP_FEATURES))}')

def shap_feature_group(feature): return next(group for group,members in SHAP_GROUPS.items() if feature in members)

shap_importance=pd.DataFrame({'feature':SHAP_FEATURES,'mean_abs_shap':np.abs(shap_values).mean(axis=0)})
shap_importance['group']=shap_importance['feature'].map(shap_feature_group)
shap_importance=shap_importance.sort_values('mean_abs_shap',ascending=False).reset_index(drop=True)
shap_importance['rank']=np.arange(1,len(shap_importance)+1)

shap_group_importance=shap_importance.groupby('group',as_index=False)['mean_abs_shap'].sum()
shap_group_importance['share_percent']=100*shap_group_importance['mean_abs_shap']/shap_group_importance['mean_abs_shap'].sum()
shap_importance.to_csv(OUT/'Fig3_source_feature_mean_abs_SHAP.csv',index=False,encoding='utf-8-sig')
shap_group_importance.to_csv(OUT/'Fig3_source_group_mean_abs_SHAP.csv',index=False,encoding='utf-8-sig')

with pd.ExcelWriter(OUT/'Fig3_source_data.xlsx') as writer:
    shap_importance.to_excel(writer,sheet_name='feature_importance',index=False)
    shap_group_importance.to_excel(writer,sheet_name='group_importance',index=False)

order=shap_importance['feature'].tolist(); ypos=np.arange(len(order))
fig,ax=plt.subplots(figsize=(FIG_WIDTH_MM/25.4/1.3,FIG_HEIGHT_MM/25.4))
ax_bar=ax.twiny(); ax_bar.set_zorder(1); ax.set_zorder(2); ax.patch.set_alpha(0)

bar_values=shap_importance['mean_abs_shap'].to_numpy()
bar_colors=[SHAP_GROUP_COLORS[group] for group in shap_importance['group']]
ax_bar.barh(ypos,bar_values,height=BAR_HEIGHT,color=bar_colors,alpha=BAR_ALPHA,edgecolor='none',zorder=1)
ax_bar.set_xlim(0,bar_values.max()*1.05); ax_bar.set_ylim(len(order)-.35,-.65); ax_bar.set_xlabel('Mean |SHAP value|',labelpad=8,fontsize=13)
ax_bar.set_yticks([]); ax_bar.tick_params(axis='x',width=.8,length=4,labelsize=9.5); ax_bar.spines['bottom'].set_visible(False); ax_bar.spines['right'].set_visible(False); ax_bar.grid(False)

for y,value,group in zip(ypos,bar_values,shap_importance['group']):
    ax_bar.text(value+bar_values.max()*.012,y,f'{value:.2f}',va='center',ha='left',fontsize=8.3,color=SHAP_GROUP_TEXT_COLORS[group],fontweight='bold',alpha=.98,zorder=2,bbox={'facecolor':'white','edgecolor':'none','alpha':.58,'pad':.25})

cmap=mpl.colormaps['coolwarm']; norm=mpl.colors.Normalize(0,1); rng=np.random.default_rng(20260611)
plot_order=[f for f in order if f in SHAP_CATEGORICAL]+[f for f in order if f in SHAP_CONTINUOUS]

for feature in plot_order:
    y=order.index(feature); j=SHAP_FEATURES.index(feature); values=shap_values[:,j]; jitter=np.clip(rng.normal(0,JITTER_SD,len(values)),-.31,.31)
    if feature in SHAP_CONTINUOUS:
        ranks=pd.to_numeric(X[feature],errors='coerce').rank(pct=True).fillna(.5).to_numpy(); colors=cmap(norm(ranks)); draw_idx=np.argsort(np.abs(ranks-.5)); point_zorder=3.5; point_alpha=POINT_ALPHA
    else:
        colors=np.tile(mpl.colors.to_rgba('#808080'),(len(values),1)); draw_idx=np.arange(len(values)); point_zorder=2.5; point_alpha=.28
    ax.scatter(values[draw_idx],(y+jitter)[draw_idx],s=POINT_SIZE,c=colors[draw_idx],alpha=point_alpha,linewidths=0,rasterized=True,zorder=point_zorder)

labels=[SHAP_DISPLAY.get(feature,feature) for feature in order]
ax.axvline(0,color='#303030',lw=.9,zorder=2)
ax.set_yticks(ypos,labels); ax.set_ylim(len(order)-.35,-.65)
ax.set_ylabel('Model input feature',labelpad=7,fontsize=13); ax.set_xlabel('SHAP value',labelpad=8,fontsize=13)
ax.grid(axis='x',color='#D9D9D9',lw=.55,linestyle=(0,(1.5,3)),zorder=0); ax.tick_params(width=.8,length=4)

category_handle=Line2D([0],[0],marker='o',color='none',markerfacecolor='#808080',markeredgecolor='none',markersize=4,label='Categorical feature')
ax.legend(handles=[category_handle],loc='lower right',bbox_to_anchor=(.992,.084),borderaxespad=0,handletextpad=.45)

cax=ax.inset_axes([.79,.030,.18,.012])
cb=mpl.colorbar.ColorbarBase(cax,cmap=cmap,norm=norm,orientation='horizontal')
cb.set_ticks([0,1]); cb.set_ticklabels(['Low','High']); cb.ax.tick_params(labelsize=8,length=0,pad=1); cb.outline.set_visible(False)
cax.set_title('Continuous feature value',fontsize=8.5,pad=2)

fig.subplots_adjust(left=.14,right=.985,bottom=.115,top=.89)
stem=OUT/'Fig3_modified_SHAP_distribution_and_importance'
fig.savefig(stem.with_suffix('.svg'),bbox_inches='tight',pad_inches=.04)
fig.savefig(stem.with_suffix('.pdf'),bbox_inches='tight',pad_inches=.04)
fig.savefig(stem.with_suffix('.png'),dpi=DPI,bbox_inches='tight',pad_inches=.04)
fig.savefig(stem.with_suffix('.tiff'),dpi=DPI,bbox_inches='tight',pad_inches=.04)
plt.show()

shap_importance

In [ ]:
from matplotlib.colors import to_rgb

DONUT_GROUPS={'Catalyst composition':['AM','AMc','Pr','Prc','Sp','Sp2','MSP'],'Operating conditions':['T','P','GHSV','Inert','H2/CO2'],'Preparation method':['PM','CT','Ct','RT','Rt','RH']}
DONUT_GROUP_ORDER=['Catalyst composition','Operating conditions','Preparation method']
DONUT_GROUP_COLORS={'Catalyst composition':'#F2C75C','Operating conditions':'#E62B1E','Preparation method':'#4D76B8'}
DONUT_TEXT_COLORS={'Catalyst composition':'#C48A00','Operating conditions':'#C90000','Preparation method':'#2E58A0'}
DONUT_GROUP_NAMES={'Catalyst composition':'Catalyst\nComposition','Operating conditions':'Operating\nConditions','Preparation method':'Preparation\nMethod'}
DONUT_DISPLAY={'H2/CO2':r'H$_2$/CO$_2$'}

mpl.rcParams.update({'font.family':'serif','font.serif':['Times New Roman','DejaVu Serif'],'mathtext.fontset':'stix','font.size':15,'axes.unicode_minus':False,'svg.fonttype':'none','pdf.fonttype':42,'savefig.facecolor':'white'})

def donut_group(feature): return next(group for group,members in DONUT_GROUPS.items() if feature in members)
def lighten(color,amount): rgb=np.array(to_rgb(color)); return tuple(rgb+(1-rgb)*amount)

donut_importance=pd.DataFrame({'feature':FEATURE_COLUMNS,'mean_abs_shap':np.abs(shap_values).mean(axis=0)})
donut_importance['group']=donut_importance['feature'].map(donut_group)
donut_importance=pd.concat([donut_importance[donut_importance['group']==group].sort_values('mean_abs_shap',ascending=False) for group in DONUT_GROUP_ORDER],ignore_index=True)

donut_group_importance=donut_importance.groupby('group',sort=False)['mean_abs_shap'].sum().reindex(DONUT_GROUP_ORDER)
donut_group_share=100*donut_group_importance/donut_group_importance.sum()
donut_importance['share_percent']=100*donut_importance['mean_abs_shap']/donut_importance['mean_abs_shap'].sum()
donut_importance.to_csv(OUT/'Fig3b_donut_feature_importance.csv',index=False,encoding='utf-8-sig')
pd.DataFrame({'group':DONUT_GROUP_ORDER,'mean_abs_shap':donut_group_importance.values,'share_percent':donut_group_share.values}).to_csv(OUT/'Fig3b_donut_group_importance.csv',index=False,encoding='utf-8-sig')

outer_values=donut_importance['mean_abs_shap'].to_numpy()
outer_features=donut_importance['feature'].tolist()
outer_groups=donut_importance['group'].tolist()
inner_values=donut_group_importance.to_numpy()

outer_colors=[]
for group in DONUT_GROUP_ORDER:
    n=(donut_importance['group']==group).sum()
    outer_colors.extend([lighten(DONUT_GROUP_COLORS[group],amount) for amount in np.linspace(.48,.72,n)])

fig,ax=plt.subplots(figsize=(8.6,8.6))
outer_wedges,_=ax.pie(outer_values,radius=1,startangle=0,counterclock=True,colors=outer_colors,wedgeprops={'width':.25,'edgecolor':'white','linewidth':2})
inner_wedges,_=ax.pie(inner_values,radius=.75,startangle=0,counterclock=True,colors=[DONUT_GROUP_COLORS[group] for group in DONUT_GROUP_ORDER],wedgeprops={'width':.13,'edgecolor':'white','linewidth':2})

external_labels=[]

for wedge,feature,group in zip(outer_wedges,outer_features,outer_groups):
    angle=(wedge.theta1+wedge.theta2)/2; span=wedge.theta2-wedge.theta1; rad=np.deg2rad(angle); label=DONUT_DISPLAY.get(feature,feature)
    if span>=8:
        rotation=angle-90 if angle<=180 else angle+90
        ax.text(.875*np.cos(rad),.875*np.sin(rad),label,ha='center',va='center',rotation=rotation,rotation_mode='anchor',fontsize=15,fontweight='bold')
    else:
        side=1 if np.cos(rad)>=0 else -1
        external_labels.append({'label':label,'angle':angle,'rad':rad,'side':side,'x':1.09*np.cos(rad),'y':1.09*np.sin(rad)})

for side in [-1,1]:
    labels_side=sorted([item for item in external_labels if item['side']==side],key=lambda item:item['y'])
    if not labels_side: continue
    min_gap=.075
    for i in range(1,len(labels_side)):
        if labels_side[i]['y']-labels_side[i-1]['y']<min_gap: labels_side[i]['y']=labels_side[i-1]['y']+min_gap
    if labels_side[-1]['y']>1.075:
        shift=labels_side[-1]['y']-1.075
        for item in labels_side: item['y']-=shift
    if labels_side[0]['y']<-1.075:
        shift=-1.075-labels_side[0]['y']
        for item in labels_side: item['y']+=shift
    for item in labels_side:
        x_text=item['x']+side*.025
        ax.annotate(item['label'],xy=(.985*np.cos(item['rad']),.985*np.sin(item['rad'])),xytext=(x_text,item['y']),ha='left' if side==1 else 'right',va='center',fontsize=13,fontweight='bold',arrowprops={'arrowstyle':'-','color':'black','linewidth':1,'shrinkA':0,'shrinkB':0,'connectionstyle':'arc3,rad=0'})

for wedge,group in zip(inner_wedges,DONUT_GROUP_ORDER):
    angle=(wedge.theta1+wedge.theta2)/2
    rad=np.deg2rad(angle)
    radius=.38 if group!='Operating conditions' else .37
    ax.text(radius*np.cos(rad),radius*np.sin(rad),f"{DONUT_GROUP_NAMES[group]}\n{donut_group_share[group]:.1f}%",ha='center',va='center',fontsize=20,color=DONUT_TEXT_COLORS[group],linespacing=1.28)

ax.set_xlim(-1.16,1.16); ax.set_ylim(-1.12,1.12); ax.set_aspect('equal'); ax.axis('off')

stem=OUT/'Fig3b_SHAP_importance_nested_donut_updated'
fig.savefig(stem.with_suffix('.svg'),bbox_inches='tight',pad_inches=.03)
fig.savefig(stem.with_suffix('.pdf'),bbox_inches='tight',pad_inches=.03)
fig.savefig(stem.with_suffix('.png'),dpi=600,bbox_inches='tight',pad_inches=.03)
fig.savefig(stem.with_suffix('.tiff'),dpi=600,bbox_inches='tight',pad_inches=.03)
plt.show()

pd.DataFrame({'group':DONUT_GROUP_ORDER,'mean_abs_shap':donut_group_importance.values,'share_percent':donut_group_share.values})

In [ ]:
from matplotlib.lines import Line2D

SHAP_GROUPS={'Catalyst composition':['AM','AMc','Pr','Prc','Sp','Sp2','MSP'],'Preparation':['PM','CT','Ct','RT','Rt','RH'],'Operating condition':['T','P','GHSV','Inert','H2/CO2']}
SHAP_GROUP_COLORS={'Operating condition':'#DB3124','Catalyst composition':'#FFDF92','Preparation':'#4B74B2'}
SHAP_TEXT_COLORS={'Operating condition':'#C82423','Catalyst composition':'#A66A00','Preparation':'#365F91'}
SHAP_DISPLAY={'H2/CO2':r'H$_2$/CO$_2$'}

def get_shap_group(feature): return next(group for group,members in SHAP_GROUPS.items() if feature in members)

shap_importance=pd.DataFrame({'feature':FEATURE_COLUMNS,'mean_abs_shap':np.abs(shap_values).mean(axis=0)})
shap_importance['group']=shap_importance['feature'].map(get_shap_group)
shap_importance=shap_importance.sort_values('mean_abs_shap',ascending=False).reset_index(drop=True)
shap_importance['rank']=np.arange(1,len(shap_importance)+1)

shap_group_importance=shap_importance.groupby('group',as_index=False)['mean_abs_shap'].sum()
shap_group_importance['share_percent']=100*shap_group_importance['mean_abs_shap']/shap_group_importance['mean_abs_shap'].sum()
shap_importance.to_csv(OUT/'Fig3_source_feature_mean_abs_SHAP.csv',index=False,encoding='utf-8-sig')
shap_group_importance.to_csv(OUT/'Fig3_source_group_mean_abs_SHAP.csv',index=False,encoding='utf-8-sig')

order=shap_importance['feature'].tolist(); ypos=np.arange(len(order))
fig3,ax=plt.subplots(figsize=(8,11))
ax_bar=ax.twiny(); ax_bar.set_zorder(1); ax.set_zorder(2); ax.patch.set_alpha(0)

bar_values=shap_importance['mean_abs_shap'].to_numpy()
bar_colors=[SHAP_GROUP_COLORS[group] for group in shap_importance['group']]
ax_bar.barh(ypos,bar_values,height=.72,color=bar_colors,alpha=.24,edgecolor='none',zorder=1)
ax_bar.set_xlim(0,bar_values.max()*1.08); ax_bar.set_ylim(len(order)-.35,-.65)
ax_bar.set_xlabel('Mean |SHAP value|',labelpad=8,fontsize=13)
ax_bar.set_yticks([]); ax_bar.tick_params(axis='x',width=.8,length=4,labelsize=9.5)
ax_bar.spines['bottom'].set_visible(False); ax_bar.spines['right'].set_visible(False); ax_bar.grid(False)

for y,value,group in zip(ypos,bar_values,shap_importance['group']):
    ax_bar.text(value+bar_values.max()*.012,y,f'{value:.2f}',va='center',ha='left',fontsize=8.3,color=SHAP_TEXT_COLORS[group],fontweight='bold',zorder=2,bbox={'facecolor':'white','edgecolor':'none','alpha':.58,'pad':.25})

cmap=mpl.colormaps['coolwarm']; norm=mpl.colors.Normalize(0,1); rng=np.random.default_rng(20260611)
plot_order=[feature for feature in order if feature in CATEGORICAL]+[feature for feature in order if feature in CONTINUOUS]

for feature in plot_order:
    y=order.index(feature); j=FEATURE_COLUMNS.index(feature); values=shap_values[:,j]
    jitter=np.clip(rng.normal(0,.105,len(values)),-.31,.31)
    if feature in CONTINUOUS:
        ranks=pd.to_numeric(X[feature],errors='coerce').rank(pct=True).fillna(.5).to_numpy()
        colors=cmap(norm(ranks)); draw_idx=np.argsort(np.abs(ranks-.5)); point_alpha=.38; point_zorder=3.5
    else:
        colors=np.tile(mpl.colors.to_rgba('#808080'),(len(values),1)); draw_idx=np.arange(len(values)); point_alpha=.28; point_zorder=2.5
    ax.scatter(values[draw_idx],(y+jitter)[draw_idx],s=8,c=colors[draw_idx],alpha=point_alpha,linewidths=0,rasterized=True,zorder=point_zorder)

ax.axvline(0,color='#303030',lw=.9,zorder=2)
ax.set_yticks(ypos,[SHAP_DISPLAY.get(feature,feature) for feature in order])
ax.set_ylim(len(order)-.35,-.65)
ax.set_ylabel('Model input feature',labelpad=7,fontsize=13)
ax.set_xlabel('SHAP value',labelpad=8,fontsize=20)
ax.grid(axis='x',color='#D9D9D9',lw=.55,linestyle=(0,(1.5,3)),zorder=0)
ax.tick_params(width=.8,length=4)

category_handle=Line2D([0],[0],marker='o',color='none',markerfacecolor='#808080',markeredgecolor='none',markersize=4,label='Categorical feature')
ax.legend(handles=[category_handle],loc='lower right',bbox_to_anchor=(.992,.084),borderaxespad=0,handletextpad=.45)

cax=ax.inset_axes([.77,.030,.20,.012])
cb=mpl.colorbar.ColorbarBase(cax,cmap=cmap,norm=norm,orientation='horizontal')
cb.set_ticks([0,1]); cb.set_ticklabels(['Low','High']); cb.ax.tick_params(labelsize=8,length=0,pad=1); cb.outline.set_visible(False)
cax.set_title('Continuous feature value',fontsize=8.5,pad=2)

fig3.subplots_adjust(left=.22,right=.985,bottom=.115,top=.89)
for suffix in ['svg','pdf']: fig3.savefig(OUT/f'Fig3_modified_SHAP_distribution_and_importance.{suffix}',bbox_inches='tight',pad_inches=.04)
fig3.savefig(OUT/'Fig3_modified_SHAP_distribution_and_importance.png',dpi=600,bbox_inches='tight',pad_inches=.04)
fig3.savefig(OUT/'Fig3_modified_SHAP_distribution_and_importance.tiff',dpi=600,bbox_inches='tight',pad_inches=.04)
plt.show()

In [ ]:
FEATURES = ['T', 'AMc', 'GHSV', 'Prc']
LABELS = {
    'T': 'Temperature (°C)',
    'AMc': 'Active metal content (wt.%)',
    'GHSV': r'Gas hourly space velocity (L/g$_{cat}$h)',
    'Prc': 'Promoter Content (wt.%) ',
}
RED, BLUE = '#C82423', '#2878B5'

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'font.size': 10,
    'axes.labelsize':14,
    'axes.titlesize': 10.5,
    'axes.linewidth': 0.8,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 9,
    'legend.frameon': False,
    'axes.spines.top': False,
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
    'savefig.facecolor': 'white',
})

raw = pd.read_excel(DATA_PATH)
X = raw[FEATURE_COLUMNS].copy()
X[CONTINUOUS] = SimpleImputer(strategy='mean').fit_transform(X[CONTINUOUS])
X[CATEGORICAL] = SimpleImputer(strategy='most_frequent').fit_transform(X[CATEGORICAL])
for column in CATEGORICAL:
    X[column] = X[column].astype(str)

model = CatBoostRegressor()
model.load_model(str(MODEL_PATH))
shap_values = np.load(SHAP_PATH, allow_pickle=True)
shap_interaction_values=np.load(INTERACTION_PATH, allow_pickle=True)
print(f'Loaded {len(X)} records, {model.tree_count_} model trees, and SHAP array {shap_values.shape}.')

In [ ]:
def counterfactual_curve(feature, grid, contexts=450, bootstrap_repeats=150, seed=10):
    sample = X.sample(min(contexts, len(X)), random_state=seed).reset_index(drop=True)
    rng = np.random.default_rng(seed)
    rows = []
    for value in grid:
        modified = sample.copy()
        modified[feature] = value
        predictions = model.predict(modified[FEATURE_COLUMNS])
        boot_means = [np.mean(predictions[rng.integers(0, len(predictions), len(predictions))]) for _ in range(bootstrap_repeats)]
        rows.append({
            'feature': feature,
            'value': float(value),
            'mean': float(np.mean(predictions)),
            'low': float(np.quantile(boot_means, 0.025)),
            'high': float(np.quantile(boot_means, 0.975)),
        })
    return pd.DataFrame(rows)

curves = []
for feature in FEATURES:
    q1, q99 = X[feature].quantile([0.01, 0.99])
    curves.append(counterfactual_curve(feature, np.linspace(q1, q99, 42), seed=10 + FEATURES.index(feature)))
curve_data = pd.concat(curves, ignore_index=True)
curve_data.to_csv(OUT / 'Fig4_abcd_counterfactual_curves_with_95CI.csv', index=False, encoding='utf-8-sig')
curve_data.head()

def smooth_shap_in_range(feature, x_range, bins=24):
    feature_idx = FEATURE_COLUMNS.index(feature)
    x = X[feature].to_numpy(float)
    y = shap_values[:, feature_idx].astype(float)
    valid = np.isfinite(x) & np.isfinite(y) & (x >= x_range[0]) & (x <= x_range[1])
    table = pd.DataFrame({'x': x[valid], 'y': y[valid]})
    table['bin'] = pd.qcut(table['x'], q=min(bins, table['x'].nunique()), duplicates='drop')
    grouped = table.groupby('bin', observed=True).median().dropna()
    x_mid, y_mid = grouped['x'].to_numpy(), grouped['y'].to_numpy()
    spline = UnivariateSpline(x_mid, y_mid, s=max(1.0, len(x_mid) * np.var(y_mid) * 0.20))
    x_fit = np.linspace(x_mid.min(), x_mid.max(), 300)
    return x_fit, spline(x_fit)

In [ ]:
# Adjustable figure parameters
FIGSIZE = (10.2, 7.4)
LINE_WIDTH = 1.8
CI_ALPHA = 0.06
SHAP_POINT_SIZE = 6
SHAP_ALPHA = 0.20
RUG_ALPHA = 0.25
DPI = 600
DISPLAY_LIMITS = {
    'T': (100, 620),
    'AMc': (0, 62),
    'GHSV': (0, 100),
    'Prc': (0, 20),
}

fig, axes = plt.subplots(2, 2, figsize=FIGSIZE)
shap_axes = []
for ax, feature, panel_label in zip(axes.flat, FEATURES, 'abcd'):
    curve = curve_data[curve_data['feature'] == feature]
    ax.plot(curve['value'], curve['mean'], color=RED, linewidth=LINE_WIDTH, label='Partial dependence')
    ax.fill_between(curve['value'], curve['low'], curve['high'], color=RED, alpha=CI_ALPHA, linewidth=0, label='95% bootstrap CI')

    feature_idx = FEATURE_COLUMNS.index(feature)
    rng = np.random.default_rng(100 + feature_idx)
    sample_idx = rng.choice(len(X), min(700, len(X)), replace=False)
    shap_ax = ax.twinx()
    shap_ax.scatter(X.iloc[sample_idx][feature], shap_values[sample_idx, feature_idx], s=SHAP_POINT_SIZE, color=BLUE, alpha=SHAP_ALPHA, linewidth=0, label='SHAP values')
    x_fit, y_fit = smooth_shap_in_range(feature, DISPLAY_LIMITS[feature])
    shap_ax.plot(x_fit, y_fit, color=BLUE, linewidth=2.0, alpha=0.95, label='SHAP fit')
    shap_ax.axhline(0, color=BLUE, linestyle=':', linewidth=0.9, alpha=0.65)
    shap_ax.set_ylabel('SHAP value (pp)', color=BLUE)
    shap_ax.tick_params(axis='y', labelcolor=BLUE)
    shap_ax.spines['right'].set_visible(True)
    shap_axes.append(shap_ax)

    rug_y = ax.get_ylim()[0]
    rug_idx = rng.choice(len(X), min(350, len(X)), replace=False)
    ax.plot(X.iloc[rug_idx][feature], np.full(len(rug_idx), rug_y), '|', color='black', alpha=RUG_ALPHA, markersize=3)
    ax.set_xlabel(LABELS[feature])
    ax.set_xlim(*DISPLAY_LIMITS[feature])
    ax.set_ylabel('Predicted CO$_2$ conversion (%)', color=RED)
    ax.tick_params(axis='y', labelcolor=RED)
    #ax.set_title(feature)
    ax.grid(color='#E5E5E5', linewidth=0.5)
    ax.text(-0.13, 1.04, panel_label, transform=ax.transAxes, fontsize=20, fontweight='bold', va='bottom')

left_handles, left_labels = axes[0, 0].get_legend_handles_labels()
right_handles, right_labels = shap_axes[0].get_legend_handles_labels()
axes[0, 0].legend(left_handles + right_handles, left_labels + right_labels, loc='upper left')
fig.subplots_adjust(left=0.09, right=0.92, bottom=0.10, top=0.96, wspace=0.42, hspace=0.40)

for suffix in ['svg', 'pdf', 'png']:
    kwargs = {'dpi': DPI} if suffix == 'png' else {}
    fig.savefig(OUT / f'Fig4_abcd_full_axis_CI.{suffix}', bbox_inches='tight', pad_inches=0.06, **kwargs)
plt.show()
print('Saved Fig4_abcd_full_axis_CI.[svg|pdf|png]')

In [ ]:
#from common import FEATURE_COLUMNS, categorical_pdp, load_model, load_shap, load_xy
mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'font.size': 8.0,
    'axes.labelsize': 8.5,
    'axes.titlesize': 9.5,
    'axes.linewidth': 0.75,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 7.5,
    'legend.fontsize': 7.2,
    'legend.frameon': False,
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
})

FEATURES = {
    'AM': ('Active metal', 7),
    'Sp': ('Primary support', 10),
    'Pr': ('Promoter', 10),
    'PM': ('Preparation method', 8),
}
COLORS = {'violin': '#D9D9D9', 'shap': '#4C78A8', 'pdp': '#D95F59', 'zero': '#4A4A4A'}

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns
from catboost import CatBoostRegressor
from sklearn.impute import SimpleImputer


def load_xy() -> tuple[pd.DataFrame, pd.Series]:
    data = pd.read_excel(DATA_PATH)
    X = data[FEATURE_COLUMNS].copy()

    X[CONTINUOUS] = SimpleImputer(
        strategy='mean'
    ).fit_transform(X[CONTINUOUS])
    X[CATEGORICAL] = SimpleImputer(
        strategy='most_frequent'
    ).fit_transform(X[CATEGORICAL])

    for column in CATEGORICAL:
        X[column] = X[column].astype(str)
    return X, data[TARGET].copy()


def load_model() -> CatBoostRegressor:
    model = CatBoostRegressor()
    model.load_model(str(MODEL_PATH))
    return model


def load_shap() -> np.ndarray:
    values = np.load(SHAP_PATH, allow_pickle=True)
    expected = (len(X), len(FEATURE_COLUMNS))
    if values.shape != expected:
        raise ValueError(f'Unexpected SHAP shape: {values.shape}; expected {expected}')
    return values


def categorical_pdp(
    model: CatBoostRegressor,
    X: pd.DataFrame,
    feature: str,
    categories: list[str] | None = None,
) -> tuple[list[str], np.ndarray]:
    # 与 common.py 默认 QUICK=False 时的计算方式一致。
    base = X.sample(len(X), random_state=10).copy()
    if categories is None:
        categories = sorted(base[feature].astype(str).unique().tolist())

    means = []
    for category in categories:
        changed = base.copy()
        changed[feature] = category
        means.append(float(np.mean(model.predict(changed[FEATURE_COLUMNS]))))
    return categories, np.asarray(means)


# 保留 common.py 导入时执行的 seaborn 白底主题。
sns.set_theme(style='white')

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'font.size': 8.0,
    'axes.labelsize': 8.5,
    'axes.titlesize': 9.5,
    'axes.linewidth': 0.75,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 7.5,
    'legend.fontsize': 7.2,
    'legend.frameon': False,
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
})

FEATURES = {
    'AM': ('Active metal', 7),
    'Sp': ('Primary support', 10),
    'Pr': ('Promoter', 10),
    'PM': ('Preparation method', 8),
}
COLORS = {
    'violin': '#D9D9D9',
    'shap': '#4C78A8',
    'pdp': '#D95F59',
    'zero': '#4A4A4A',
}

X, _ = load_xy()
model = load_model()
shap_values = load_shap()


In [ ]:
def prepare_feature(feature, top_n):
    counts = X[feature].astype(str).value_counts()
    all_categories = counts.index.tolist()
    categories, pdp_abs = categorical_pdp(model, X, feature, all_categories)
    pdp_map = dict(zip(categories, pdp_abs))
    weighted_reference = np.average(
        [pdp_map[c] for c in all_categories],
        weights=[counts[c] for c in all_categories],
    )
    idx = FEATURE_COLUMNS.index(feature)
    frame = pd.DataFrame({'category': X[feature].astype(str), 'shap': shap_values[:, idx]})
    summaries = frame.groupby('category')['shap'].agg(['mean', 'median']).join(counts.rename('n'))
    summaries['pdp_centered'] = [pdp_map[c] - weighted_reference for c in summaries.index]
    selected = counts.head(top_n).index.tolist()
    summaries = summaries.loc[selected]
    summaries['consensus'] = (summaries['mean'] + summaries['pdp_centered']) / 2
    summaries = summaries.sort_values('consensus', ascending=True)
    distributions = [frame.loc[frame['category'].eq(c), 'shap'].to_numpy() for c in summaries.index]
    return summaries, distributions

prepared = {feature: prepare_feature(feature, top_n) for feature, (_, top_n) in FEATURES.items()}


In [ ]:
def chemical_label(value):
    return re.sub(r'(\d+)', r'$_{\1}$', str(value))

fig, axes = plt.subplots(2, 2, figsize=(9.2, 7.3))
panel_letters = 'abcd'

for ax, (feature, (title, _)), letter in zip(axes.flat, FEATURES.items(), panel_letters):
    summary, distributions = prepared[feature]
    y = np.arange(len(summary))
    values = np.concatenate(distributions)
    extent = max(abs(np.quantile(values, 0.01)), abs(np.quantile(values, 0.99)),
                 abs(summary['pdp_centered']).max()) * 1.16

    violins = ax.violinplot(distributions, positions=y, vert=False, widths=0.66,
                            showmeans=False, showmedians=False, showextrema=False,
                            points=120, bw_method=0.28)
    for body in violins['bodies']:
        body.set_facecolor(COLORS['violin'])
        body.set_edgecolor('none')
        body.set_alpha(0.58)

    for yi, dist in zip(y, distributions):
        q25, q75 = np.quantile(dist, [0.25, 0.75])
        ax.hlines(yi, q25, q75, color='#888888', linewidth=1.0, zorder=3)

    for yi, shap_mean, pdp_effect in zip(y, summary['mean'], summary['pdp_centered']):
        ax.plot([shap_mean, pdp_effect], [yi, yi], color='#B7B7B7', linewidth=0.7, zorder=3)

    ax.scatter(summary['mean'], y, s=25, color=COLORS['shap'], edgecolor='white',
               linewidth=0.45, zorder=4)
    ax.scatter(summary['pdp_centered'], y, s=34, marker='D', facecolor='white',
               edgecolor=COLORS['pdp'], linewidth=1.2, zorder=5)
    for yi, shap_mean, pdp_effect in zip(y, summary['mean'], summary['pdp_centered']):
        ax.annotate(f'{shap_mean:+.1f}', (shap_mean, yi), xytext=(-2, 5), textcoords='offset points',
                    ha='right', va='bottom', fontsize=6.3, color=COLORS['shap'], zorder=6)
        ax.annotate(f'{pdp_effect:+.1f}', (pdp_effect, yi), xytext=(2, -6), textcoords='offset points',
                    ha='left', va='top', fontsize=6.3, color=COLORS['pdp'], zorder=6)

    ax.axvline(0, color=COLORS['zero'], linewidth=0.75, linestyle='--', zorder=1)
    ax.set_xlim(-extent, extent)
    ax.set_yticks(y, [chemical_label(c) for c in summary.index])
    ax.set_xlabel('Mean SHAP effect (pp)', color=COLORS['shap'])
    ax.tick_params(axis='x', colors=COLORS['shap'])
    top_axis = ax.secondary_xaxis('top', functions=(lambda x: x, lambda x: x))
    top_axis.set_xlabel('Centered PDP effect (pp)', color=COLORS['pdp'], labelpad=4)
    top_axis.tick_params(axis='x', colors=COLORS['pdp'], labelsize=7.5)
    top_axis.spines['top'].set_color(COLORS['pdp'])
    ax.grid(axis='x', color='#E8E8E8', linewidth=0.5, zorder=0)
    ax.text(-0.16, 1.03, letter, transform=ax.transAxes, fontsize=10, fontweight='bold', va='bottom')

legend_handles = [
    Line2D([0], [0], color='#A0A0A0', linewidth=6, alpha=0.45, label='SHAP distribution'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor=COLORS['shap'],
           markeredgecolor='white', markersize=5.5, label='Mean SHAP effect'),
    Line2D([0], [0], marker='D', color='none', markerfacecolor='white',
           markeredgecolor=COLORS['pdp'], markersize=5.5, label='Centered PDP effect'),
]
fig.legend(
    handles=legend_handles,
    loc='lower center',
    bbox_to_anchor=(0.5, 0.015),
    ncol=3,
    fontsize=11,
    handlelength=2.2,
    handletextpad=0.7,
    columnspacing=1.8,
    borderaxespad=0.0,
)
fig.subplots_adjust(left=0.13, right=0.98, bottom=0.12, top=0.95, wspace=0.34, hspace=0.55)

stem = OUT / 'Fig5_compact_PDP_SHAP_revision'
fig.savefig(stem.with_suffix('.svg'), bbox_inches='tight')
fig.savefig(stem.with_suffix('.pdf'), bbox_inches='tight')
fig.savefig(stem.with_suffix('.png'), dpi=400, bbox_inches='tight')
plt.show()


In [ ]:
df = raw.copy()
dx1=df.iloc[:,[3,4,5,6,7,9,11,12,13,14,15,17,18,19,20,21,22,26]]
dy1=df.iloc[:,27]

In [ ]:
continuous_features = ['AMc', 'Prc','MSP',  'CT', 'Ct', 'RT','Rt', 'RH', 'T', 'P', 'GHSV', 'Inert', 'H2/CO2']
discrete_features=['AM',  'Pr',  'Sp', 'Sp2', 'PM']
cont_imputer = SimpleImputer(strategy='mean')
dx1[continuous_features] = cont_imputer.fit_transform(dx1[continuous_features])
disc_imputer = SimpleImputer(strategy='most_frequent')
dx1[discrete_features] = disc_imputer.fit_transform(dx1[discrete_features])


In [ ]:
plt.rcParams.update({
    'font.family':'serif',
    'font.serif':'Times New Roman',
    'mathtext.fontset':'stix',
    'axes.unicode_minus':False
})

fig=plt.figure(figsize=(12,6),dpi=300)
shap.summary_plot(shap_interaction_values,dx1,plot_type='bar',show=False,max_display=20)
ax=plt.gca()
for patch in ax.patches: patch.set_facecolor('#4B74B2')
plt.savefig(OUT / 'shap_interaction-summary.svg', format='svg', bbox_inches='tight', pad_inches=.1)
plt.show()

In [ ]:
mean_shap=np.abs(shap_interaction_values).mean(0)
df=pd.DataFrame(mean_shap,index=dx1.columns,columns=dx1.columns)

df.where(df.values==np.diagonal(df),df.values,inplace=True)

In [ ]:
size1=18
columns=['AM','AMc','Pr','Prc','Sp','Sp2','MSP','PM','CT','Ct','RT','Rt','RH','T','P','GHSV','Inert',r'H$_2$/CO$_2$']

mpl.rcParams.update({'font.family':'serif','font.serif':['Times New Roman','DejaVu Serif'],'mathtext.fontset':'stix','font.size':size1,'axes.linewidth':1.0,'axes.unicode_minus':False,'svg.fonttype':'none','pdf.fonttype':42,'savefig.facecolor':'white'})

values=df.to_numpy(float)
cmap=mpl.colormaps['coolwarm']
norm=mpl.colors.Normalize(vmin=0,vmax=2.5)

fig,ax=plt.subplots(figsize=(11,10))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

for i in range(len(df.columns)):
    for j in range(len(df.columns)):
        value=df.iloc[i,j]
        color=cmap(norm(value))
        if i<j:
            ax.scatter(j,i,s=10+abs(value)*500,color=color,alpha=.95,edgecolor='white',linewidth=.35)
        elif i>j:
            ax.text(j,i,f'{value:.2f}',ha='center',va='center',color=color,fontsize=11)

ax.set_xticks(range(len(columns)))
ax.set_xticklabels(columns,rotation=45,ha='right',fontsize=14)
ax.set_yticks(range(len(columns)))
ax.set_yticklabels(columns,fontsize=14)
ax.set_xlim(-.6,len(columns)-.4)
ax.set_ylim(-.6,len(columns)-.4)
ax.set_aspect('equal')
ax.tick_params(axis='both',length=3,width=.8)

for side in ['left','right','top','bottom']:
    ax.spines[side].set_visible(True)
    ax.spines[side].set_linewidth(1.0)
    ax.spines[side].set_color('black')

sm=mpl.cm.ScalarMappable(cmap=cmap,norm=norm)
sm.set_array([])
cbar=fig.colorbar(sm,ax=ax,pad=.045,fraction=.045)
cbar.set_label('Interaction strength',fontsize=16)
cbar.ax.tick_params(labelsize=14,width=.8)
cbar.outline.set_linewidth(.8)

plt.tight_layout()
plt.savefig(OUT/'Interaction_strength_updated.svg',bbox_inches='tight',pad_inches=.08)
plt.savefig(OUT/'Interaction_strength_updated.png',dpi=600,bbox_inches='tight',pad_inches=.08)
plt.show()

In [ ]:
from matplotlib import patches
interaction_only=np.array(shap_interaction_values,copy=True)
idx=np.arange(interaction_only.shape[1])
interaction_only[:,idx,idx]=0

plt.rcParams.update({
    'font.family':'serif',
    'font.serif':['Times New Roman','DejaVu Serif'],
    'mathtext.fontset':'stix',
    'axes.unicode_minus':False,
    'svg.fonttype':'none',
    'pdf.fonttype':42
})

fig=plt.figure(figsize=(12,2),dpi=300)

shap.summary_plot(
    interaction_only,
    dx1,
    max_display=20,
    plot_type='compact_dot',
    cmap='coolwarm',
    show=False
)
fig=plt.gcf()
fig.set_size_inches(6,10)
fig.subplots_adjust(left=.30,right=.88)
ax_main=plt.gcf().axes[0]
rect=patches.Rectangle(
    (0,0),1,1,
    transform=ax_main.transAxes,
    fill=False,
    linewidth=1,
    edgecolor='black',
    clip_on=False
)
ax_main.add_patch(rect)

plt.savefig(
    OUT/'shap_interaction_only_40.svg',
    format='svg',
    bbox_inches='tight',
    pad_inches=.1
)
plt.savefig(
    OUT/'shap_interaction_only_40.png',
    dpi=600,
    bbox_inches='tight',
    pad_inches=.1
)
plt.show()

In [ ]:
interaction_array=np.asarray(shap_interaction_values); n_features=len(FEATURE_COLUMNS)
if interaction_array.ndim!=3 or interaction_array.shape[0]!=len(X): raise ValueError(f'Interaction array shape {interaction_array.shape} does not match {len(X)} records')
if interaction_array.shape[1]<n_features or interaction_array.shape[2]<n_features: raise ValueError(f'Interaction array shape {interaction_array.shape} is smaller than {n_features} features')
interaction_array=interaction_array[:,:n_features,:n_features].astype(float)

TOP_N, SAMPLE_N, SEED=20,1200,0
interaction_strength=np.mean(np.abs(interaction_array),axis=0)
interaction_strength=(interaction_strength+interaction_strength.T)/2

interaction_matrix=pd.DataFrame(interaction_strength,index=FEATURE_COLUMNS,columns=FEATURE_COLUMNS)
interaction_matrix.to_csv(OUT/'SHAP_interaction_strength_matrix.csv',encoding='utf-8-sig')

interaction_long=pd.DataFrame([{'feature_1':FEATURE_COLUMNS[i],'feature_2':FEATURE_COLUMNS[j],'interaction_strength':interaction_strength[i,j],'type':'main effect' if i==j else 'interaction'} for i in range(n_features) for j in range(n_features)])
interaction_long.to_csv(OUT/'SHAP_interaction_strength_long.csv',index=False,encoding='utf-8-sig')

top_pairs=sorted([(i,j,interaction_strength[i,j]) for i in range(n_features) for j in range(n_features) if i!=j],key=lambda x:x[2],reverse=True)[:TOP_N]
top_pair_summary=pd.DataFrame([{'rank':k+1,'feature_1':FEATURE_COLUMNS[i],'color_feature':FEATURE_COLUMNS[j],'pair':f'{FEATURE_COLUMNS[i]} * {FEATURE_COLUMNS[j]}','mean_abs_interaction':value} for k,(i,j,value) in enumerate(top_pairs)])
top_pair_summary.to_csv(OUT/'SHAP_top_directed_interactions.csv',index=False,encoding='utf-8-sig')
top_pair_summary

In [ ]:
def valid_domain_grid(
    observed: pd.DataFrame,
    x_col: str,
    y_col: str,
    x_values: np.ndarray,
    y_values: np.ndarray,
    quantile: float = 0.90,
    neighbors: int = 25,
) -> tuple[np.ndarray, np.ndarray]:

    scaler = StandardScaler()

    obs_scaled = scaler.fit_transform(
        observed[[x_col, y_col]]
    )

    k = min(neighbors, len(observed))

    nn = NearestNeighbors(
        n_neighbors=k
    ).fit(obs_scaled)

    obs_dist, _ = nn.kneighbors(obs_scaled)

    threshold = np.quantile(
        obs_dist[:, -1],
        quantile
    )

    points = np.array([
        (x, y)
        for y in y_values
        for x in x_values
    ])

    point_dist, _ = nn.kneighbors(
        scaler.transform(points)
    )

    distance = point_dist[:, -1].reshape(
        len(y_values),
        len(x_values)
    )

    valid = distance <= threshold

    return valid, distance

In [ ]:
import sys
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import gaussian_filter, gaussian_filter1d
GRID_N, CONTEXTS, EXPORT_DPI = 55, 350, 600
SELECTED_T, TARGET_LEVELS = [200,250,300,350,400], [40,50,60,70]

def make_surface(y_feature):
    xg=np.linspace(X['T'].quantile(.03),X['T'].quantile(.97),GRID_N); yg=np.linspace(X[y_feature].quantile(.03),X[y_feature].quantile(.97),GRID_N)
    contexts=X.sample(min(CONTEXTS,len(X)),random_state=10).reset_index(drop=True); z=np.empty((GRID_N,GRID_N))
    for yi,yv in enumerate(yg):
        batch=pd.concat([contexts]*GRID_N,ignore_index=True); batch['T']=np.repeat(xg,len(contexts)); batch[y_feature]=yv
        z[yi]=model.predict(batch[FEATURE_COLUMNS]).reshape(GRID_N,len(contexts)).mean(axis=1)
    valid,distance=valid_domain_grid(X,'T',y_feature,xg,yg,quantile=.90)
    return xg,yg,np.clip(z,0,100),valid,distance

x_amc,y_amc,z_amc,v_amc,d_amc=make_surface('AMc')
pd.DataFrame([{'T':xv,'AMc':yv,'prediction':z_amc[yi,xi],'valid_domain':bool(v_amc[yi,xi]),'support_distance':d_amc[yi,xi]} for yi,yv in enumerate(y_amc) for xi,xv in enumerate(x_amc)]).to_csv(OUT/'Fig7_source_T_AMc_surface.csv',index=False,encoding='utf-8-sig')

x_ghsv,y_ghsv,z_ghsv,v_ghsv,d_ghsv=make_surface('GHSV')
pd.DataFrame([{'T':xv,'GHSV':yv,'prediction':z_ghsv[yi,xi],'valid_domain':bool(v_ghsv[yi,xi]),'support_distance':d_ghsv[yi,xi]} for yi,yv in enumerate(y_ghsv) for xi,xv in enumerate(x_ghsv)]).to_csv(OUT/'Fig7_source_T_GHSV_surface.csv',index=False,encoding='utf-8-sig')

In [ ]:
line_colors=dict(zip(SELECTED_T,mpl.colormaps['coolwarm'](np.linspace(.05,.95,len(SELECTED_T)))))
body_cmap=mpl.colors.LinearSegmentedColormap.from_list('body',['#4B74B2','#B7CBE5','#FFDF92','#E98B5B','#DB3124'])

mpl.rcParams.update({'font.family':'serif','font.serif':['Times New Roman','DejaVu Serif'],'mathtext.fontset':'stix','font.size':16,'axes.labelsize':20,'xtick.labelsize':16,'ytick.labelsize':16,'axes.linewidth':.8,'axes.spines.top':False,'axes.spines.right':False,'legend.fontsize':13,'legend.title_fontsize':14,'legend.frameon':False,'svg.fonttype':'none','pdf.fonttype':42})

def draw_map(ax,xg,yg,z,valid,y_feature):
    masked=np.ma.masked_where(~valid,gaussian_filter(z,.4)); cf=ax.contourf(xg,yg,masked,levels=np.linspace(0,80,17),cmap=body_cmap,vmin=0,vmax=80)
    for target in TARGET_LEVELS:
        cs=ax.contour(xg,yg,masked,levels=[target],colors='black',linewidths=1.55); ax.clabel(cs,fmt={target:f'{target}%'},fontsize=11.5,inline=True,colors='black')
    ax.contourf(xg,yg,(~valid).astype(int),levels=[.5,1.5],colors=['#E3E3E3'],alpha=.52,zorder=8)
    ax.set_xlim(xg.min(),xg.max()); ax.set_ylim(yg.min(),yg.max()); ax.set_xlabel('Temperature (°C)'); ax.set_ylabel('Active metal content (wt.%)' if y_feature=='AMc' else 'Gas hourly space velocity (L/g$_{cat}$h)')
    return cf

def draw_curves(ax,xg,yg,z,valid,y_feature,leverage,threshold):
    interp=RegularGridInterpolator((yg,xg),gaussian_filter(z,.4),bounds_error=False,fill_value=np.nan); vinterp=RegularGridInterpolator((yg,xg),valid.astype(float),method='nearest',bounds_error=False,fill_value=0)
    curve=np.linspace(yg.min(),yg.max(),220); rows=[]
    for t in SELECTED_T:
        pts=np.column_stack([curve,np.full_like(curve,t)]); frame=pd.DataFrame({'T':t,y_feature:curve,'prediction':interp(pts),'supported':vinterp(pts)>.5}); rows.append(frame); frame=frame[frame['supported']]
        groups=frame.index.to_series().diff().fillna(1).ne(1).cumsum()
        for k,(_,seg) in enumerate(frame.groupby(groups)): ax.plot(seg[y_feature],gaussian_filter1d(seg['prediction'].to_numpy(),1),color=line_colors[t],lw=2.2,label=f'{t} °C' if k==0 else None)
    ax.axvspan(*leverage,color='#FFDF92',alpha=.16); ax.axvspan(threshold,yg.max(),color='#BDBDBD',alpha=.07)
    for target in [50,60,70]: ax.axhline(target,color='#B8B8B8',lw=.7,ls=':',zorder=0)
    ax.set_xlim(yg.min(),yg.max()); ax.set_ylim(0,82); ax.set_xlabel('Active metal content (wt.%)' if y_feature=='AMc' else 'Gas hourly space velocity (L/g$_{cat}$h)'); ax.set_ylabel('Predicted CO$_2$ conversion (%)'); ax.grid(color='#DFDFDF',lw=.5,ls=(0,(1.5,3))); ax.legend(loc='lower right',title='Temperature')
    return pd.concat(rows,ignore_index=True)

fig,axes=plt.subplots(2,2,figsize=(14,11.5)); fig.subplots_adjust(left=.08,right=.91,bottom=.09,top=.96,wspace=.36,hspace=.42)
cf=draw_map(axes[0,0],x_amc,y_amc,z_amc,v_amc,'AMc'); curves_amc=draw_curves(axes[0,1],x_amc,y_amc,z_amc,v_amc,'AMc',(5,20),20)
draw_map(axes[1,0],x_ghsv,y_ghsv,z_ghsv,v_ghsv,'GHSV'); curves_ghsv=draw_curves(axes[1,1],x_ghsv,y_ghsv,z_ghsv,v_ghsv,'GHSV',(y_ghsv.min(),10),10)
for ax,label in zip(axes.flat,'abcd'): ax.text(-.14,1.03,label,transform=ax.transAxes,fontsize=20,fontweight='bold')
cax=fig.add_axes([.935,.20,.020,.62]); cb=fig.colorbar(cf,cax=cax); cb.set_label('Predicted CO$_2$ conversion (%)',fontsize=15); cb.ax.tick_params(labelsize=12)
for suffix,kwargs in {'png':{'dpi':EXPORT_DPI},'tiff':{'dpi':EXPORT_DPI},'pdf':{},'svg':{}}.items(): fig.savefig(OUT/f'Fig7_dual_factor_operation_maps_revision.{suffix}',bbox_inches='tight',pad_inches=.08,**kwargs)
curves_amc.to_csv(OUT/'Fig7_source_T_AMc_paths.csv',index=False,encoding='utf-8-sig'); curves_ghsv.to_csv(OUT/'Fig7_source_T_GHSV_paths.csv',index=False,encoding='utf-8-sig')
plt.show()

In [ ]:

METALS = ['Ni','Co','Ru','Fe','Rh','Pd','Metal-free']
METAL_COLORS = {'Ni':'#B33A3A','Co':'#D47A35','Ru':'#5B78A6','Fe':'#738FB8','Rh':'#7868A6','Pd':'#5F91A8','Metal-free':'#55916B'}
PEAK_METALS = {'Ni','Co'}
METAL_INTERPRETATION = {'Ni':'Interior maximum','Co':'Probable interior maximum','Ru':'Rising toward upper boundary','Fe':'Rising within observed range','Rh':'Rising within observed range','Pd':'Rising within observed range','Metal-free':'Rising within observed range'}

if len(raw) != len(shap_values): raise ValueError(f'Data rows {len(raw)} do not match SHAP rows {len(shap_values)}')

metal_data = raw[['Reference','AM','T']].copy()
metal_data['SHAP_T'] = shap_values[:,FEATURE_COLUMNS.index('T')]

def aggregate_metal(metal):
    metal_raw = metal_data[metal_data['AM'].astype(str).eq(metal)][['Reference','T','SHAP_T']].dropna().copy()
    metal_agg = metal_raw.groupby(['Reference','T'],as_index=False).agg(SHAP_T=('SHAP_T','median'),n_rows=('SHAP_T','size'))
    return metal_raw,metal_agg

def fit_metal_curve(metal_agg,grid):
    coef = np.polyfit(metal_agg['T'].to_numpy(float),metal_agg['SHAP_T'].to_numpy(float),2)
    return np.polyval(coef,grid),coef

def reference_bootstrap(metal_agg,grid,iterations=500,seed=2026):
    references = metal_agg['Reference'].astype(str).unique()
    if len(references) < 2: return None,None,np.array([])
    rng = np.random.default_rng(seed); curves_boot,vertices = [],[]
    source = metal_agg.copy(); source['Reference'] = source['Reference'].astype(str)
    for _ in range(iterations):
        sampled = rng.choice(references,size=len(references),replace=True)
        boot = pd.concat([source[source['Reference'].eq(ref)] for ref in sampled],ignore_index=True)
        try:
            curve_boot,coef_boot = fit_metal_curve(boot,grid); curves_boot.append(curve_boot)
            if coef_boot[0] < 0: vertices.append(-coef_boot[1]/(2*coef_boot[0]))
        except Exception: continue
    curves_boot = np.asarray(curves_boot)
    return np.quantile(curves_boot,.025,axis=0),np.quantile(curves_boot,.975,axis=0),np.asarray(vertices)

metal_results,metal_plot_objects = [],{}
for metal in METALS:
    metal_raw,metal_agg = aggregate_metal(metal)
    t_min,t_max = float(metal_agg['T'].min()),float(metal_agg['T'].max())
    grid = np.linspace(t_min,t_max,260)
    curve,coef = fit_metal_curve(metal_agg,grid)
    low,high,vertices = reference_bootstrap(metal_agg,grid)
    peak = float(-coef[1]/(2*coef[0])) if coef[0] < 0 else np.nan
    if metal in PEAK_METALS and len(vertices):
        valid_vertices = vertices[(vertices>=t_min)&(vertices<=t_max)]
        peak_low,peak_high = np.quantile(vertices,[.025,.975]); inside_fraction = len(valid_vertices)/len(vertices)
    else: peak_low = peak_high = inside_fraction = np.nan
    metal_results.append({'metal':metal,'n_rows':len(metal_raw),'n_reference_temperature_points':len(metal_agg),'n_references':metal_raw['Reference'].nunique(),'T_min':t_min,'T_max':t_max,'fit_type':'reference-balanced quadratic','peak_T':peak,'peak_T_bootstrap_low':peak_low,'peak_T_bootstrap_high':peak_high,'bootstrap_peak_inside_fraction':inside_fraction,'interpretation':METAL_INTERPRETATION[metal]})
    metal_plot_objects[metal] = {'raw':metal_raw,'agg':metal_agg,'grid':grid,'curve':curve,'low':low,'high':high}

metal_diagnostics = pd.DataFrame(metal_results)
metal_diagnostics.to_csv(OUT/'Fig8_fit_diagnostics.csv',index=False,encoding='utf-8-sig')
metal_diagnostics.to_excel(OUT/'Fig8_fit_diagnostics.xlsx',index=False)
metal_diagnostics

In [ ]:
mpl.rcParams.update({'font.family':'serif','font.serif':['Times New Roman','Times','DejaVu Serif'],'mathtext.fontset':'stix','font.size':8.5,'axes.labelsize':12,'axes.titlesize':10,'xtick.labelsize':10,'ytick.labelsize':10,'axes.linewidth':.65,'axes.spines.top':False,'axes.spines.right':False,'svg.fonttype':'none','pdf.fonttype':42,'savefig.facecolor':'white'})

def metal_equation_text(coef):
    a,b,c = coef
    return rf'$y={a*10000:.2f}t^2{b*100:+.2f}t{c:+.1f}$'

def metal_markers(coef,t_min,t_max):
    roots = np.roots(coef)
    inside_roots = sorted(root.real for root in roots if abs(root.imag)<1e-8 and t_min<=root.real<=t_max)
    threshold = inside_roots[-1] if inside_roots else np.nan
    vertex = -coef[1]/(2*coef[0]) if coef[0]<0 else np.nan
    return threshold,vertex if np.isfinite(vertex) and t_min<=vertex<=t_max else t_max

fig8,axes8 = plt.subplots(len(METALS),1,figsize=(4,6),sharex=True,gridspec_kw={'hspace':.045,'left':.10,'right':.985,'bottom':.14,'top':.985})
for i,(ax,metal) in enumerate(zip(axes8,METALS)):
    obj = metal_plot_objects[metal]; metal_raw,metal_agg = obj['raw'],obj['agg']; grid,curve = obj['grid'],obj['curve']
    diag = metal_diagnostics.loc[metal_diagnostics['metal'].eq(metal)].iloc[0]
    coef = np.polyfit(metal_agg['T'].to_numpy(float),metal_agg['SHAP_T'].to_numpy(float),2)
    color = METAL_COLORS[metal]
    sample = metal_raw.sample(min(len(metal_raw),650),random_state=12+i)
    ax.scatter(sample['T'],sample['SHAP_T'],s=2.2,color='#505862',alpha=.1,linewidth=0,rasterized=True)
    ax.scatter(metal_agg['T'],metal_agg['SHAP_T'],s=5+1.5*np.sqrt(metal_agg['n_rows'].to_numpy(float)),facecolor=color,edgecolor='white',linewidth=.2,alpha=.24,zorder=2)
    if obj['low'] is not None: ax.fill_between(grid,obj['low'],obj['high'],color=color,alpha=.075,linewidth=0,zorder=1)
    ax.plot(grid,curve,color=color,lw=1.55,zorder=3)
    threshold,marker_t = metal_markers(coef,float(diag['T_min']),float(diag['T_max']))
    ax.plot([diag['T_min'],diag['T_max']],[0,0],color='#B7B7B7',lw=.7,linestyle=(0,(2.2,2.2)),zorder=0)
    if np.isfinite(threshold): ax.scatter([threshold],[0],s=20,facecolor='white',edgecolor=color,linewidth=1.1,zorder=4)
    ax.scatter([marker_t],[np.polyval(coef,marker_t)],s=22,facecolor=color,edgecolor='white',linewidth=.55,zorder=4)
    ax.text(.985,.68,metal,transform=ax.transAxes,ha='right',va='center',fontsize=10,fontweight='bold',color=color,bbox={'facecolor':'white','edgecolor':'none','alpha':.8,'pad':.7})
    ax.text(.985,.28,metal_equation_text(coef),transform=ax.transAxes,ha='right',va='center',fontsize=8,color='#5F5F5F',bbox={'facecolor':'white','edgecolor':'none','alpha':.8,'pad':.6})
    ax.set_xlim(0,850); ax.set_yticks([0]); ax.set_yticklabels(['']); ax.tick_params(axis='y',length=2.5,pad=2)
    ax.grid(axis='x',color='#F0F0F0',lw=.38); ax.spines['left'].set_color('#A5A5A5'); ax.spines['bottom'].set_visible(True)
    ax.spines['bottom'].set_color('#E2E2E2' if i<len(METALS)-1 else '#222222'); ax.spines['bottom'].set_linewidth(.55 if i<len(METALS)-1 else .85)
    if i<len(METALS)-1: ax.tick_params(axis='x',length=0,labelbottom=False)

axes8[-1].set_xlabel(r'Temperature ($^\circ$C)')
fig8.text(.042,.54,'SHAP value for temperature (pp)',rotation=90,ha='center',va='center',fontsize=12)
fig8_stem = OUT/'Fig8_metal_SHAP_response_ridgeline_preview'
fig8.savefig(fig8_stem.with_suffix('.svg'),bbox_inches='tight',pad_inches=.04)
fig8.savefig(fig8_stem.with_suffix('.pdf'),bbox_inches='tight',pad_inches=.04)
fig8.savefig(fig8_stem.with_suffix('.png'),dpi=600,bbox_inches='tight',pad_inches=.04)
fig8.savefig(fig8_stem.with_suffix('.tiff'),dpi=600,bbox_inches='tight',pad_inches=.04)
plt.show()

In [ ]:
source_rows = []
for feature, (summary, _) in prepared.items():
    table = summary.reset_index().rename(columns={'index': 'category'})
    table.insert(0, 'feature', feature)
    source_rows.append(table)
pd.concat(source_rows, ignore_index=True).to_csv(OUT / 'Fig5_compact_summary.csv', index=False)
print(f'Saved outputs to: {OUT}')


In [ ]:
REFERENCES = {'Ni':'Al2O3', 'Ru':'TiO2'}
MIN_SUPPORT = 20; N_BOOT = 250; SEED = 20260614
RED, BLUE, GRAY = '#DB3124', '#4B74B2', '#9A9A9A'

mpl.rcParams.update({
 'font.family':'serif', 'font.serif':['Times New Roman','DejaVu Serif'], 'mathtext.fontset':'stix',
 'font.size':12, 'axes.labelsize':13, 'axes.titlesize':13, 'xtick.labelsize':11,
 'ytick.labelsize':11, 'axes.linewidth':.85, 'axes.spines.top':False, 'axes.spines.right':False,
 'svg.fonttype':'none', 'pdf.fonttype':42, 'savefig.facecolor':'white'
})
FEATURES = ['AM','AMc','Pr','Prc','Sp','Sp2','MSP','PM','CT','Ct','RT','Rt','RH','T','P','GHSV','Inert','H2/CO2']
CATEGORICAL = ['AM','Pr','Sp','Sp2','PM']; CONTINUOUS = [x for x in FEATURES if x not in CATEGORICAL]
raw = pd.read_excel(DATA_PATH); X = raw[FEATURES].copy()
ci, ca = SimpleImputer(strategy='mean'), SimpleImputer(strategy='most_frequent')
X[CONTINUOUS] = ci.fit_transform(X[CONTINUOUS]); X[CATEGORICAL] = ca.fit_transform(X[CATEGORICAL])
for c in CATEGORICAL: X[c] = X[c].astype(str)
#model = CatBoostRegressor()
#model.load_model(str(MODEL_PATH))
print(f'Loaded {len(X)} records and {model.tree_count_} model trees.')

In [ ]:
REFERENCES = {'Ni':'Al2O3', 'Ru':'TiO2'}
MIN_SUPPORT = 20; N_BOOT = 250; SEED = 0
rng = np.random.default_rng(SEED)
support_counts = X.groupby(['AM','Sp']).size()
rows = []

for metal, reference in REFERENCES.items():
    metal_contexts = X[X['AM'] == metal].sample(min(800, (X['AM'] == metal).sum()), random_state=SEED).reset_index(drop=True)
    alternatives = [sp for (am,sp), n in support_counts.items() if am == metal and n >= MIN_SUPPORT and sp != reference]
    for support in alternatives:
        boot_effects = []
        for _ in range(N_BOOT):
            context = metal_contexts.iloc[rng.integers(0, len(metal_contexts), len(metal_contexts))].reset_index(drop=True)
            candidate = context.copy(); candidate['AM'] = metal; candidate['Sp'] = support
            baseline = context.copy(); baseline['AM'] = metal; baseline['Sp'] = reference
            boot_effects.append(float(np.mean(model.predict(candidate[FEATURES]) - model.predict(baseline[FEATURES]))))
        effect = float(np.mean(boot_effects))
        sign_consistency = float(np.mean(np.sign(boot_effects) == np.sign(effect)))
        rows.append({
            'metal':metal, 'support':support, 'reference_support':reference,
            'label':f'{metal}–{support}', 'effect':effect,
            'low':float(np.quantile(boot_effects,.025)), 'high':float(np.quantile(boot_effects,.975)),
            'sign_consistency':sign_consistency, 'support_count':int(support_counts[(metal,support)]),
            'stable_80pct':bool(sign_consistency >= .80)
        })

effects = pd.DataFrame(rows)

In [ ]:
effects.sort_values(['metal','effect'])
effects

In [ ]:
counts = raw.groupby(['AM','Sp']).size().reset_index(name='support_count')
counts = counts[counts['AM'].isin(['Ni','Ru'])].sort_values(['AM','support_count'], ascending=[True,False])
references = counts.groupby('AM', as_index=False).first().rename(columns={'Sp':'reference_support'})
reference_map = dict(zip(references['AM'], references['reference_support']))
print('References selected by maximum within-metal AM-Sp sample count:', reference_map)
print(counts.to_string(index=False))

In [ ]:
# Display every alternative meeting the AM-support minimum count, then add explicit reference rows.
ni = effects[effects['metal']=='Ni'].sort_values('effect')
ru = effects[effects['metal']=='Ru'].sort_values('effect')
alternative_selected = pd.concat([ni, ru]).copy()

reference_rows = []
for _, row in references.iterrows():
    metal, support, n = row['AM'], row['reference_support'], int(row['support_count'])
    reference_rows.append({
        'metal':metal, 'support':support, 'reference_support':support,
        'label':f'{metal}–{support}', 'effect':0.0, 'low':0.0, 'high':0.0,
        'sign_consistency':np.nan, 'support_count':n, 'stable_80pct':False,
        'is_reference':True, 'selection_rule':'most frequent primary support within active metal'
    })

alternative_selected['is_reference'] = False
alternative_selected['selection_rule'] = np.where(
    alternative_selected['metal'].eq('Ni'),
    'all Ni alternatives with AM-Sp support count >= 20',
    'all Ru alternatives with AM-Sp support count >= 20'
)
selected = pd.concat([alternative_selected, pd.DataFrame(reference_rows)], ignore_index=True)
selected['ci_excludes_zero'] = (selected['low'] > 0) | (selected['high'] < 0)
selected['display_state'] = np.select(
    [selected['is_reference'], ~selected['ci_excludes_zero'], selected['effect'] > 0],
    ['reference', 'interval crosses zero', 'positive'], default='negative'
)
selected.to_csv(OUT/'Fig4c_source_selected_effects_with_references.csv', index=False, encoding='utf-8-sig')
counts.to_csv(OUT/'Fig4c_source_reference_support_counts.csv', index=False, encoding='utf-8-sig')
selected[['metal','support','reference_support','effect','low','high','support_count','sign_consistency','display_state','selection_rule']]


In [ ]:
def chem(text):
    return (text.replace('Al2O3','Al$_2$O$_3$').replace('CeO2','CeO$_2$').replace('ZrO2','ZrO$_2$')
                .replace('TiO2','TiO$_2$').replace('SiO2','SiO$_2$').replace('Cr2O3','Cr$_2$O$_3$')
                .replace('Y2O3','Y$_2$O$_3$'))

plot_rows, group_bounds, y0 = [], {}, 0
for metal in ['Ni', 'Ru']:
    group = selected[selected['metal']==metal].sort_values('effect').copy()
    group['y'] = y0 + np.arange(len(group))
    plot_rows.append(group)
    group_bounds[metal] = (group['y'].min(), group['y'].max())
    y0 = group['y'].max()+2.6
plot_df = pd.concat(plot_rows, ignore_index=True)

fig, ax = plt.subplots(figsize=(7.45, 7.7))
max_alt_n = plot_df.loc[~plot_df['is_reference'], 'support_count'].max()

for _, row in plot_df.iterrows():
    if row['is_reference']:
        ax.scatter(0, row['y'], s=72, marker='D', facecolor='white', edgecolor=GRAY, linewidth=1.5, zorder=4)
        ax.text(.48, row['y'], f"reference; n={int(row['support_count'])}", va='center', fontsize=8.5, color='#666666')
        continue
    color = GRAY if not row['ci_excludes_zero'] else (RED if row['effect'] > 0 else BLUE)
    size = 40 + 105*np.sqrt(row['support_count']/max_alt_n)
    ax.errorbar(row['effect'], row['y'], xerr=[[row['effect']-row['low']],[row['high']-row['effect']]],
                fmt='none', ecolor=color, elinewidth=1.5, capsize=3, zorder=2)
    ax.scatter(row['effect'], row['y'], s=size, color=color, edgecolor='white', linewidth=.7, zorder=3)
    ax.text(row['high']+.18, row['y'], f"n={int(row['support_count'])}; {row['sign_consistency']:.0%} ",
            va='center', fontsize=9, color='#666666')

ax.axvline(0, color='black', linewidth=.9, zorder=1)
separator_y = (group_bounds['Ni'][1]+group_bounds['Ru'][0])/2
ax.axhline(separator_y, color='#BFBFBF', linewidth=.8, linestyle='--')
ax.set_yticks(plot_df['y'], [chem(x) for x in plot_df['label']])
ax.tick_params(axis='y', labelsize=12)
ax.invert_yaxis()
ax.set_xlabel('$\\Delta$ predicted CO$_2$ conversion (pp)')
ax.grid(axis='x', color='#D9D9D9', linewidth=.5, alpha=.75)
span = plot_df['high'].max()-plot_df['low'].min()
ax.set_xlim(plot_df['low'].min()-.1*span, plot_df['high'].max()+.2*span)
ax.set_ylim(plot_df['y'].max()+.75, plot_df['y'].min()-1.35)
ax.text(-.025, group_bounds['Ni'][0]-.55, 'Ni-based systems', transform=ax.get_yaxis_transform(), ha='right', va='center', fontsize=13, fontweight='bold')
ax.text(-.025, group_bounds['Ru'][0]-.55, 'Ru-based systems', transform=ax.get_yaxis_transform(), ha='right', va='center', fontsize=13, fontweight='bold')
ax.text(-.13, 1.02, 'a', transform=ax.transAxes, fontsize=20, fontweight='bold', va='bottom')

legend_handles = [
    Line2D([0],[0], marker='o', linestyle='', markerfacecolor=RED, markeredgecolor='white', label='Positive effect', markersize=6.5),
    Line2D([0],[0], marker='o', linestyle='', markerfacecolor=BLUE, markeredgecolor='white', label='Negative effect', markersize=6.5),
    Line2D([0],[0], marker='o', linestyle='', markerfacecolor=GRAY, markeredgecolor='white', label='95% interval crosses zero', markersize=6.5),
    Line2D([0],[0], marker='D', linestyle='', markerfacecolor='white', markeredgecolor=GRAY, label='Reference', markersize=5.8)
]
ax.legend(handles=legend_handles, loc='upper right', ncol=1, frameon=False,
          borderaxespad=.55, columnspacing=1.0, handletextpad=.4)
fig.subplots_adjust(left=.30, right=.96, bottom=.10, top=.97)

for suffix in ['svg','pdf']:
    fig.savefig(OUT/f'Fig4c_counterfactual_all_Ni.{suffix}', bbox_inches='tight', pad_inches=.06)
fig.savefig(OUT/'Fig4c_counterfactual_all_Ni.png', dpi=500, bbox_inches='tight', pad_inches=.06)
plt.show()


In [ ]:
# User-specified main colors and low-saturation extensions.
COLORS = {
    'Ni-Al2O3': '#4B74B2',
    'Ni-CeO2': '#DB3124',
    'Ru-TiO2': '#8567A9',
    'Ru-CB': '#E88980',
}

mpl.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'stix', 'font.size': 16, 'axes.labelsize': 18,
    'axes.titlesize': 18, 'axes.linewidth': 1.1, 'xtick.labelsize': 16,
    'ytick.labelsize': 18, 'legend.fontsize': 13.5, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False,
    'svg.fonttype': 'none', 'pdf.fonttype': 42, 'savefig.facecolor': 'white'
})

In [ ]:
SUBSETS = [('Ni','Al2O3'), ('Ni','CeO2'), ('Ru','TiO2'), ('Ru','CB')]
COLORS = {'Ni-Al2O3':'#4B74B2', 'Ni-CeO2':'#DB3124', 'Ru-TiO2':'#8567A9', 'Ru-CB':'#E88980'}
rng = np.random.default_rng(20260614)
curve_rows, summary_rows = [], []

for metal, support in SUBSETS:
    subset = X[(X['AM']==metal)&(X['Sp']==support)].copy()
    n = len(subset)
    if n == 0: continue
    t_min, t_max = subset['T'].quantile([.05,.95])
    contexts = subset.sample(min(260,n), random_state=14).reset_index(drop=True)
    key = f'{metal}-{support}'
    summary_rows.append({'subset':key, 'n_records':n, 'n_promoter_levels':subset['Pr'].nunique(), 'n_preparation_methods':subset['PM'].nunique(), 'n_metal_loading_levels':subset['AMc'].nunique(), 'median_AMc_wt_pct':subset['AMc'].median(), 'T_5th_C':t_min, 'T_95th_C':t_max})
    for temperature in np.linspace(t_min,t_max,34):
        changed = contexts.copy(); changed['T'] = temperature
        predictions = model.predict(changed[FEATURE_COLUMNS])
        boot = [predictions[rng.integers(0,len(predictions),len(predictions))].mean() for _ in range(100)]
        curve_rows.append({'subset':key, 'metal':metal, 'support':support, 'T':temperature, 'mean':predictions.mean(), 'low':np.quantile(boot,.025), 'high':np.quantile(boot,.975), 'n_records':n})

curves = pd.DataFrame(curve_rows); summary = pd.DataFrame(summary_rows)
curves.to_csv(OUT/'Fig4b_source_conditional_temperature_curves.csv', index=False, encoding='utf-8-sig')
summary.to_csv(OUT/'Fig4b_subset_heterogeneity_summary.csv', index=False, encoding='utf-8-sig')
summary

In [ ]:
# Draw a vertically enlarged, simplified panel b.
fig, ax = plt.subplots(figsize=(7, 10))

for _, row in summary.iterrows():
    key = row['subset']; g = curves[curves['subset'] == key]
    metal = key.split('-')[0]
    linestyle = '-' if metal == 'Ni' else '--'
    label = key.replace('Al2O3','Al$_2$O$_3$').replace('CeO2','CeO$_2$').replace('ZrO2','ZrO$_2$').replace('TiO2','TiO$_2$')
    mean_s = g['mean'].rolling(5, center=True, min_periods=1).mean()
    low_s = g['low'].rolling(5, center=True, min_periods=1).mean()
    high_s = g['high'].rolling(5, center=True, min_periods=1).mean()
    ax.plot(g['T'], mean_s, color=COLORS[key], lw=2.5, linestyle=linestyle, label=label)
    ax.fill_between(g['T'], low_s, high_s, color=COLORS[key], alpha=.11, linewidth=0)

ax.set_ylabel('CO$_2$ conversion (%)')
#ax.set_title('Conditional temperature responses\nof heterogeneous metal–support subsets', pad=16)
ax.text(-.18, 1.035, 'b', transform=ax.transAxes, fontsize=22, fontweight='bold', va='bottom')
ax.grid(color='#D9D9D9', linewidth=.45, alpha=.72)
ax.legend(ncol=1, loc='upper left', handlelength=2.5, borderaxespad=.45)
ax.set_xlabel('Temperature (°C)', fontsize=22)
ax.set_ylim(-3, 84)

fig.savefig(OUT/'Fig4b_four_curves_large_font.svg', bbox_inches='tight', pad_inches=.06)
fig.savefig(OUT/'Fig4b_four_curves_large_font.pdf', bbox_inches='tight', pad_inches=.06)
fig.savefig(OUT/'Fig4b_four_curves_large_font.png', dpi=500, bbox_inches='tight', pad_inches=.06)
plt.show()

note = {
    'interpretation': 'Curves are conditional means for heterogeneous database-defined metal-support subsets, not fixed catalyst formulations.',
    'solid_lines': 'Ni-associated subsets', 'dashed_lines': 'Ru-associated subsets',
    'Pr': 'number of promoter levels within subset', 'PM': 'number of preparation methods within subset'
}
(OUT/'Fig4b_interpretation_note.json').write_text(json.dumps(note, indent=2), encoding='utf-8')
note